# MF-HGNN MDD_LeaveGroupOut Reproduction Guide

This notebook includes:

1. Data loading and a brief overview
2. Dataset splitting (stratified K-fold) and summary
3. Full training procedure (calling `main.py`)
4. Evaluation with the best saved models and viewing logs

All core training and evaluation logic is identical to the original experimental code used in the paper, implemented via `main.py`, `dataload.py`, `model.py`, etc.


In [1]:
# Environment and dependencies

import sys, os
import numpy as np
import torch

from opt import OptInit
from dataload import dataloader

# Initialize hyperparameters (same as running main.py from the command line)
opt = OptInit().initialize()
print('Device:', opt.device)

# Print several important paths for this notebook
print('subject_IDs_path:', opt.subject_IDs_path)
print('phenotype_path:', opt.phenotype_path)
print('data_path:', opt.data_path)
print('log_path:', opt.log_path)
print('ckpt_path:', opt.ckpt_path)


 Using GPU in torch
==========       CONFIG      =============
train:1
use_cpu:False
lr:0.01
wd:5e-05
num_iter:400
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./
log_path:./inffus_log.txt
subject_IDs_path:/mnt/restmdd_ho_841_2/MDD_pcp/subject_IDs.txt
phenotype_path:/mnt/restmdd_ho_841_2/MDD_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/restmdd_ho_841_2/MDD_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> Phase is train.
 Using GPU in torch
==========       CONFIG      =============
train:1
use_cpu:False
lr:0.01
wd:5e-05
num_iter:400
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./
log_path:./inffus_log.txt
subject_IDs_path:/mnt/restmdd_ho_841_2/MDD_pcp/subject_IDs.txt
phenotype_path:/mnt/restmdd_ho_841_2/MDD_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/restmdd_ho_841_2/MDD_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END   

## 1. Data loading and overview

In this section, we use the `dataloader` class in `dataload.py` to reproduce exactly the same data-loading pipeline as in the official experiments, and briefly inspect key data structures:

- Raw node features `raw_features`
- Label vector `y`
- Phenotypic data `phonetic_data` (site, sex, age, etc.)
- Dynamic functional connectivity `dynamic_fc`


In [2]:
# 1. Data loading (detailed prints)

dl = dataloader()
raw_features, y, nonimg, phonetic_score, time_series, dynamic_fc = dl.load_data()

# Avoid flooding the notebook output; only show parts of large arrays
np.set_printoptions(threshold=20)

# Basic information
print('Number of raw_features samples:', len(raw_features))
print('Shape of labels y:', np.array(y).shape)
print('Shape of non-image phenotypes nonimg:', np.array(nonimg).shape)
print('Number of dynamic_fc samples:', len(dynamic_fc))

# Inspect the first sample
first_feat = raw_features[0]
print('\nType of the first sample:', type(first_feat))
if hasattr(first_feat, 'shape'):
    print('Shape of the first sample features:', first_feat.shape)
elif hasattr(first_feat, 'x'):
    # For torch_geometric.data.Data
    print('Shape of node features x of the first sample:', first_feat.x.shape)
    print('Shape of edge_index of the first sample:', first_feat.edge_index.shape)

# Label distribution
unique, counts = np.unique(y, return_counts=True)
print('\nLabel distribution (label: count):')
for u, c in zip(unique, counts):
    print(f'  {u}: {c}')

# Phenotypic information (SITE_ID, SEX, AGE_AT_SCAN)
print('\nSITE_ID for all samples:')
print(phonetic_score['SITE_ID'])

print('\nSEX for all samples:')
print(phonetic_score['SEX'])

print('\nAGE_AT_SCAN for all samples:')
print(phonetic_score['AGE_AT_SCAN'])


Number of raw_features samples: 841
Shape of labels y: (841,)
Shape of non-image phenotypes nonimg: (841, 3)
Number of dynamic_fc samples: 841

Type of the first sample: <class 'torch_geometric.data.data.Data'>
Shape of node features x of the first sample: torch.Size([112, 112])
Shape of edge_index of the first sample: torch.Size([2, 504])

Label distribution (label: count):
  0.0: 457
  1.0: 384

SITE_ID for all samples:
[0. 0. 0. ... 2. 2. 2.]

SEX for all samples:
[1. 2. 1. ... 2. 1. 1.]

AGE_AT_SCAN for all samples:
[26. 26. 43. ... 74. 78. 63.]


## 2. Dataset splitting strategy in the current code

In this repository, dataset splitting is implemented in `dataload.py` via `dataloader.data_split()`. The key idea is:

- Perform subject-level cross-validation.
- Use `StratifiedKFold`, stratifying on the **joint distribution of diagnosis label DX_GROUP and site SITE_ID**.
- Concretely, `labels` and `sites` are combined into `strat_labels = labels * 1000 + sites`, and stratified sampling is performed on `strat_labels`.
- As a result, in each fold, the proportions of different sites and diagnosis categories are as close as possible to the full dataset.

The following code cell calls `dataloader.data_split` from this directory and prints, for each fold, the number of samples and the distribution of `(label, SITE_ID)` combinations, providing a clear view of the actual splitting strategy used in this code.


In [3]:
# 2*. Detailed illustration of the current splitting strategy

n_folds = opt.n_folds
cv_splits = dl.data_split(n_folds)

labels = np.array(y)
sites = np.array(phonetic_score['SITE_ID'])

print(f'Total number of samples: {len(labels)}')
print(f'Number of folds n_folds: {n_folds}\n')

for fold, (train_idx, test_idx) in enumerate(cv_splits):
    y_tr, s_tr = labels[train_idx], sites[train_idx]
    y_te, s_te = labels[test_idx], sites[test_idx]

    print(f'====== Fold {fold} ======')
    print(f'Number of training samples: {len(train_idx)}, number of test samples: {len(test_idx)}')

    # Distribution of (label, SITE_ID) combinations in training and test sets
    comb_tr = y_tr * 1000 + s_tr
    comb_te = y_te * 1000 + s_te
    u_tr, c_tr = np.unique(comb_tr, return_counts=True)
    u_te, c_te = np.unique(comb_te, return_counts=True)

    print('Training set (label*1000 + SITE_ID) combination distribution:')
    print(dict(zip(u_tr.astype(int), c_tr)))
    print('Test set (label*1000 + SITE_ID) combination distribution:')
    print(dict(zip(u_te.astype(int), c_te)))
    print()


Total number of samples: 841
Number of folds n_folds: 10

====== Fold 0 ======
Number of training samples: 756, number of test samples: 85
Training set (label*1000 + SITE_ID) combination distribution:
{0: 253, 1: 78, 2: 80, 1000: 226, 1001: 63, 1002: 56}
Test set (label*1000 + SITE_ID) combination distribution:
{0: 29, 1: 8, 2: 9, 1000: 25, 1001: 7, 1002: 7}

====== Fold 1 ======
Number of training samples: 757, number of test samples: 84
Training set (label*1000 + SITE_ID) combination distribution:
{0: 253, 1: 78, 2: 80, 1000: 226, 1001: 63, 1002: 57}
Test set (label*1000 + SITE_ID) combination distribution:
{0: 29, 1: 8, 2: 9, 1000: 25, 1001: 7, 1002: 6}

====== Fold 2 ======
Number of training samples: 757, number of test samples: 84
Training set (label*1000 + SITE_ID) combination distribution:
{0: 254, 1: 78, 2: 80, 1000: 225, 1001: 63, 1002: 57}
Test set (label*1000 + SITE_ID) combination distribution:
{0: 28, 1: 8, 2: 9, 1000: 26, 1001: 7, 1002: 6}

====== Fold 3 ======
Number of

## 3. Training example (calling main.py)

The official training and evaluation pipeline is implemented in `main.py` under `if __name__ == "__main__":`, which includes:

- Initializing hyperparameters (`OptInit`)
- Calling `dataloader` to load the data
- Running stratified K-fold cross-validation
- Printing training and test metrics at each epoch
- Saving the best model (on the test set) for each fold to `opt.ckpt_path`

In this notebook we invoke `main.py` via a command-line call so that the behavior is exactly the same as running the script directly, and a **training log file** is automatically generated (handled by the `Logger` class).

> Training from scratch can be time-consuming, so in practice we usually pre-train on the server, save the best models and logs, and then use this notebook mainly for demonstration.


In [ ]:
import sys

# Use the Python interpreter of the current notebook kernel
!{sys.executable} main.py --train 1 --num_iter 400 --n_folds 10 --ckpt_path ./ckpt_demo/ --log_path ./run.log


In [4]:
# 3.2 View a snippet of the training log

log_path = './run.log'

if os.path.exists(log_path):
    print(f'Reading log file: {log_path}\n')
    with open(log_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    # Show only the last 40 lines for readability
    for line in lines[-40:]:
        print(line.rstrip())
else:
    print('Log file does not exist. Please run the training cell above first.')


Reading log file: ./run.log

Epoch: 372,	ce loss: 0.01049,	ce loss_cla: 0.01049,	train acc: 0.99736,	test acc: 0.97619,	test spe: 1.00000,	test sen: 0.94872      2026-03-24 06:21:13
Epoch: 373,	ce loss: 0.01019,	ce loss_cla: 0.01019,	train acc: 0.99604,	test acc: 0.97619,	test spe: 1.00000,	test sen: 0.94872      2026-03-24 06:22:10
Epoch: 374,	ce loss: 0.00239,	ce loss_cla: 0.00239,	train acc: 1.00000,	test acc: 0.97619,	test spe: 1.00000,	test sen: 0.94872      2026-03-24 06:23:06
Epoch: 375,	ce loss: 0.00527,	ce loss_cla: 0.00527,	train acc: 0.99736,	test acc: 0.98810,	test spe: 1.00000,	test sen: 0.97436      2026-03-24 06:24:03
Epoch: 376,	ce loss: 0.00887,	ce loss_cla: 0.00887,	train acc: 0.99604,	test acc: 0.98810,	test spe: 1.00000,	test sen: 0.97436      2026-03-24 06:24:59
Epoch: 377,	ce loss: 0.00593,	ce loss_cla: 0.00593,	train acc: 0.99604,	test acc: 0.98810,	test spe: 1.00000,	test sen: 0.97436      2026-03-24 06:25:55
Epoch: 378,	ce loss: 0.01507,	ce loss_cla: 0.01507,	t

## 4. Evaluation with the best saved models

In `main.py`, the `evaluate()` function is responsible for:

- Loading the best-performing model for each fold from `opt.ckpt_path`.
- Running forward passes on the corresponding test set.
- Computing and printing multiple metrics (accuracy, AUC, sensitivity, specificity, F1-score, etc.).

Here we call the same script with `--train` set to `0` to perform evaluation using the saved models.

> Before evaluation, make sure that the corresponding model checkpoints for all folds have been successfully saved in `ckpt_path` during training.


In [5]:
import sys

# 4.1 Example: evaluate using pre-trained models

# Assume the training stage used ./ckpt_demo/ as ckpt_path.
# Here we keep the same ckpt_path, set train to 0, and run evaluation only.

!{sys.executable} main.py --train 0 --num_iter 400 --n_folds 10 --ckpt_path ./ckpt_demo/ --log_path ./test.log


 Using GPU in torch
==========       CONFIG      =============
train:0
use_cpu:False
lr:0.01
wd:5e-05
num_iter:400
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./ckpt_demo/
log_path:./test.log
subject_IDs_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
==========     CONFIG END    =============


===> Phase is eval.
 Using GPU in torch
==========       CONFIG      =============
train:0
use_cpu:False
lr:0.01
wd:5e-05
num_iter:400
dropout:0.3
num_classes:2
n_folds:10
ckpt_path:./ckpt_demo/
log_path:./test.log
subject_IDs_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/subject_IDs.txt
phenotype_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/Phenotypic_V1_0b_preprocessed1.csv
data_path:/mnt/restmdd_ho_841_2/ABIDE_pcp/cpac/filt_noglobal
alpha:0.65
beta:1.5
k1:0.9
k2:0.5
time:260326
device:cuda:0
=========

In [6]:
# 4.2 View a snippet of the evaluation log

log_path_test = './test.log'

if os.path.exists(log_path_test):
    print(f'Reading evaluation log file: {log_path_test}\n')
    with open(log_path_test, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    # Show the last 40 lines of evaluation results
    for line in lines[-40:]:
        print(line.rstrip())
else:
    print('Evaluation log file does not exist. Please run the evaluation cell above first.')


Reading evaluation log file: ./test.log

    (convs4): ModuleList(
      (0): TransformerConv(2020, 20, heads=1)
      (1-4): 4 x TransformerConv(40, 20, heads=1)
    )
    (bns): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (bns2): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (bns3): ModuleList(
      (0-4): 5 x BatchNorm1d(20, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (out_fc): Linear(in_features=100, out_features=2, bias=True)
    (conv5): Linear(in_features=20, out_features=1, bias=False)
    (conv6): Linear(in_features=20, out_features=1, bias=False)
    (conv7): Linear(in_features=20, out_features=1, bias=False)
    (conv8): Linear(in_features=20, out_features=1, bias=False)
    (conv9): Linear(in_features=20, out_features=1, bias=False)
    (conv10): Linear(in_features=20, out_features=1, bias=False)
 